# PDF RAG with Chroma — Persisting Vectors for Real (5 of 6)

## RAG Workshop Series

This notebook is part of a 6-notebook series (split from the original `RAG.ipynb`), each one runnable on its own in Google Colab:

1. **`basic_RAG.ipynb`** — naive vector RAG: chunk → embed → cosine similarity → prompt → LLM
2. **`hybrid_search_RAG.ipynb`** — BM25 keyword search + semantic search fusion + cross-encoder reranking
3. **`query_and_chunking_RAG.ipynb`** — query rewriting, advanced chunking strategies, metadata filtering
4. **`agentic_RAG.ipynb`** — Corrective RAG (CRAG), Adaptive RAG (routing), Agentic RAG (ReAct loop)
5. **`pdf_chroma_RAG.ipynb`** — build RAG over a real PDF, store vectors persistently in Chroma
6. **`rag_when_to_use.ipynb`** — reference: when RAG is (and isn't) the right tool

Each notebook installs its own dependencies and rebuilds whatever context it needs, so you can open any one directly without running the others first.

## Why this notebook exists

Every notebook so far has stored embeddings in a plain Python list or NumPy array. That works for a demo, but it has a real problem: **restart the runtime and every embedding is gone.** Re-computing embeddings for a 10-page PDF is a few seconds. Re-computing them for a 10,000-page manual, every single time someone reopens the notebook, is not something anyone would actually do.

This notebook builds RAG over an actual PDF and stores the vectors in [Chroma](https://docs.trychroma.com/), a local, embedded vector database. No server, no account, no API key for the vector store itself — just a folder on disk that survives a restart.

## What you'll do

1. Generate a sample PDF (or upload your own) and extract its text
2. Chunk it, keeping track of which page each chunk came from
3. Store the chunks + embeddings in a persistent Chroma collection
4. Query it
5. **Prove persistence** — reconnect to the same collection without re-embedding anything
6. Filter by metadata (page number) — the thing `query_and_chunking_RAG.ipynb` Step 13 set up but never actually used
7. Close the loop: retrieved chunks → prompt → LLM → grounded answer

## Setup

Needs an `NSCALE_API` key as a Colab secret (🔑 icon in the left sidebar) for the final answer step — same as the other notebooks.

In [ ]:
!pip install -q chromadb pypdf fpdf2 sentence-transformers langchain-text-splitters openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get('NSCALE_API'),
    base_url="https://inference.api.nscale.com/v1",
)

In [ ]:
%%capture
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

## Step 1: Make a real PDF (or bring your own)

We'll reuse the same AWS services knowledge base from `agentic_RAG.ipynb` — one page per service — and write it out as an actual `aws_guide.pdf`. This is the exact file the metadata example in `query_and_chunking_RAG.ipynb` Step 13 was pretending to have.

**Prefer to use your own PDF instead?** Skip the generation cell below, run:
```python
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
```
then continue from Step 1's text-extraction cell — every step after this one works on any PDF, unchanged.

In [ ]:
SERVICES = [
    ("Amazon S3", """Amazon S3 is an object storage service designed for storing
and retrieving files.

It is commonly used for backups, static websites, data lakes,
application assets, and large collections of unstructured data."""),
    ("Amazon SQS", """Amazon SQS is a managed message queue service.

It allows applications to communicate asynchronously.
Producers send messages to a queue and consumers process
those messages independently.

SQS is commonly used to decouple distributed applications
so that one service does not have to wait for another service
to finish processing."""),
    ("AWS Lambda", """AWS Lambda is a serverless compute service.

Developers upload code and AWS executes the code in response
to events. Lambda automatically manages servers and scales
applications based on incoming requests."""),
    ("Amazon DynamoDB", """Amazon DynamoDB is a managed NoSQL database.

It provides low-latency access to data and automatically
scales to handle large workloads.

DynamoDB is commonly used for applications that require
fast access to large amounts of structured data."""),
    ("Amazon API Gateway", """Amazon API Gateway is a managed service for creating,
publishing, monitoring, and securing APIs.

It can route incoming HTTP requests to backend services
such as AWS Lambda."""),
    ("Amazon SNS", """Amazon SNS is a publish-subscribe messaging service.

It allows applications to send notifications to multiple
subscribers or downstream systems.

SNS is commonly used for event broadcasting and notifications."""),
    ("AWS CloudWatch", """AWS CloudWatch provides monitoring and observability for
AWS resources and applications.

It collects metrics, logs, and events and can trigger
alarms when specified conditions occur."""),
    ("AWS IAM", """AWS IAM manages access to AWS resources.

It allows administrators to define users, roles, policies,
and permissions controlling which resources applications
and users can access.""")
]

In [ ]:
from fpdf import FPDF

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

for name, text in SERVICES:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, name, ln=True)
    pdf.set_font("Helvetica", size=12)
    pdf.multi_cell(0, 8, text.strip())

pdf_path = "aws_guide.pdf"
pdf.output(pdf_path)

print(f"Wrote {pdf_path}")

Now extract the text back out — this is the same first step you'd take with any PDF.

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

print("Pages:", len(reader.pages))
print()
print(reader.pages[0].extract_text()[:300])

## Step 2: Chunk it, keeping page numbers

We use `RecursiveCharacterTextSplitter` (same as `query_and_chunking_RAG.ipynb`) page by page, so every chunk knows which page it came from — that page number becomes metadata we can filter on later.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

pdf_chunks = []      # list[str]
pdf_metadatas = []   # list[dict], one per chunk

for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text()

    for chunk in splitter.split_text(page_text):
        pdf_chunks.append(chunk)
        pdf_metadatas.append({"page": page_number, "source": pdf_path})

print("Number of chunks:", len(pdf_chunks))
print()
print("Sample chunk (page", pdf_metadatas[0]["page"], "):")
print(pdf_chunks[0])

## Step 3: Store in Chroma

We wrap our already-loaded `SentenceTransformer` as Chroma's embedding function, so the vectors are identical to the ones used everywhere else in this series — Chroma's default would otherwise try to call OpenAI's embedding API, which needs a different key.

In [ ]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings


class SentenceTransformerEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        return embedding_model.encode(input).tolist()


chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="aws_guide",
    embedding_function=SentenceTransformerEmbeddingFunction(),
)

collection.add(
    ids=[f"chunk-{i}" for i in range(len(pdf_chunks))],
    documents=pdf_chunks,
    metadatas=pdf_metadatas,
)

print("Chunks stored in Chroma:", collection.count())

## Step 4: Query it

Same sanity check as `basic_RAG.ipynb`: does the SQS-flavoured question surface the right chunk?

In [ ]:
question = "Which AWS service can help decouple applications?"

results = collection.query(
    query_texts=[question],
    n_results=3,
)

for text, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(f"page {metadata['page']} | distance {distance:.3f}")
    print(text[:200])
    print()

## Step 5: Prove persistence

This is the part a plain Python list can't do. Pretend the Colab runtime just restarted — fresh client, fresh Python process, **no re-embedding**. We just reconnect to the same folder on disk.

In [ ]:
# Simulates a fresh runtime: a brand-new client pointing at the same path.
reconnected_client = chromadb.PersistentClient(path="./chroma_db")

reconnected_collection = reconnected_client.get_collection(
    name="aws_guide",
    embedding_function=SentenceTransformerEmbeddingFunction(),
)

print("Chunks still there:", reconnected_collection.count())

results = reconnected_collection.query(
    query_texts=["Which AWS service can help decouple applications?"],
    n_results=1,
)

print(results["documents"][0][0][:200])

No `SentenceTransformer.encode()` call happened in that cell — the vectors were already on disk. That's the difference a vector store buys you: `basic_RAG.ipynb`'s `chunk_embeddings` array would be gone the moment the runtime restarted.

## Step 6: Metadata filtering, for real this time

`query_and_chunking_RAG.ipynb` Step 13 defined a `metadata_filter` dict and then never used it — that was the exercise. Here's the working version, using Chroma's `where=` argument to search only page 2 (the SQS page).

In [ ]:
results = collection.query(
    query_texts=["How do applications communicate without waiting for each other?"],
    n_results=3,
    where={"page": 2},
)

for text, metadata in zip(results["documents"][0], results["metadatas"][0]):
    print(f"page {metadata['page']}")
    print(text[:200])
    print()

## Step 7: Close the loop — retrieve, prompt, answer

Same `build_prompt` pattern as `basic_RAG.ipynb`, but the context now comes from Chroma instead of an in-memory array.

In [ ]:
def build_prompt(question, retrieved_chunks):

    context = "\n\n".join(
        f"(page {m['page']}) {text}"
        for text, m in retrieved_chunks
    )

    return f"""
Answer the question using only the context below. Cite the page number.

CONTEXT:

{context}

QUESTION:

{question}
"""


question = "Which AWS service should I use to decouple applications?"

results = collection.query(query_texts=[question], n_results=3)

retrieved = list(zip(results["documents"][0], results["metadatas"][0]))

prompt = build_prompt(question, retrieved)

response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    max_tokens=300,
    temperature=0.2,
)

print(response.choices[0].message.content)

## Stretch tasks (fast finishers)

Pick one — both reuse everything above unchanged:

- **Bring your own PDF.** Use the `files.upload()` snippet from Step 1, rerun every cell from text extraction onward, and see if retrieval quality holds up on a document the corpus wasn't tuned for.
- **Add BM25 on top of Chroma.** `hybrid_search_RAG.ipynb` built a hybrid semantic+BM25 search over an in-memory list — do the same thing here, but pull the candidate documents from `collection.get()` instead of a Python list, and fuse BM25 scores with Chroma's cosine distances.

---

➡️ **Next:** [`rag_when_to_use.ipynb`](./rag_when_to_use.ipynb) — now that you've built naive, hybrid, agentic, and persistent RAG, the reference for deciding when any of this is actually worth reaching for.